1. Define necessary transformation for video

In [6]:
import torchvision.transforms.v2 as transforms
import torch

# Parameters
root_dir = './action_classification_dataset/train'
classes = ['Biking', 'Diving', 'HorseRiding']
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))

])

In [7]:
import numpy as np
import cv2

def load_video(path, resize, max_frames):
        cap = cv2.VideoCapture(path)
        frames = []
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                if resize:
                    frame = cv2.resize(frame, resize)
                frames.append(frame)
                if max_frames and len(frames) >= max_frames:
                    break
        finally:
            cap.release()
        return np.array(frames)

2. Load a Convolutional Neural Network

In [8]:
from network.convlstm import ConvLSTM

import torch.nn as nn

class ConvLSTMModel(nn.Module):
    def __init__(self, num_classes, image_size, batch_size=1, hidden_dim=[64, 64]):
        super(ConvLSTMModel, self).__init__()
        """
        input_dim: Number of channels in input
        hidden_dim: Number of hidden channels
        kernel_size: Size of kernel in convolutions
        num_layers: Number of LSTM layers stacked on each other
        batch_first (default=False): Whether or not dimension 0 is the batch or not
        bias (default=True): Bias or no bias in Convolution
        return_all_layers(default=False): Return the list of computations for all layers
        Note: Will do same padding.

        """
        self.batch_size = batch_size
        self.convlstm = ConvLSTM(input_dim=3, hidden_dim=hidden_dim, kernel_size=(3, 3), num_layers=len(hidden_dim), batch_first=True)
        self.fc = nn.Linear(image_size[0] * image_size[1] * hidden_dim[-1], num_classes)

    def forward(self, x):
        # dada shape (batch_size, time, c, h, w)
        _, last_state_list = self.convlstm(x)
    
        # reshape information from last hidden state
        x = last_state_list[-1][0].view(self.batch_size, -1)
        x = self.fc(x)
        return x

In [9]:
# Parameters
num_classes = len(classes)
model = ConvLSTMModel(num_classes, image_size=(112, 112))
model.load_state_dict(torch.load('./video_classification_model.pt'))

<All keys matched successfully>

3. Evaluate the network

In [10]:
max_frames = 30
resize = (112, 112)

video_path = 'action_classification_dataset/test/Diving/v_Diving_g15_c01.avi'
video_frames = load_video(video_path, resize, max_frames)
video_frames = [transform(frame) for frame in video_frames]

# Stack frames to create a tensor
video = torch.stack(video_frames)

# Check if GPU (cuda) is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Move our model to device (GPU or CPU)
model.to(device)
model.eval()

video = video.to(device)
video = video.unsqueeze(0)

outputs = model(video)
outputs = outputs.squeeze(0)

score = torch.softmax(outputs, 0)
_, predicted = torch.max(score, 0)

print(f'Predicted: {classes[predicted]} {score[predicted] * 100:.02f}%')

Predicted: Diving 99.30%
